In [2]:
from collections import Counter
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
import os 
from dotenv import load_dotenv

import csv
from typing import List, Dict
from trulens.apps.custom import TruCustomApp
from trulens.core import TruSession
from trulens.core import Feedback
from trulens.providers.openai import OpenAI as tru_openai
from trulens.apps.custom import instrument


In [3]:
from trulens.core import TruSession


session = TruSession()

# Uncomment the following to reset database 
# session.reset_database()

🦑 Initialized with db url sqlite:///default.sqlite .
🛑 Secret keys may be written to the database. See the `database_redact_keys` option of `TruSession` to prevent this.


In [4]:
load_dotenv()

True

In [5]:
client = OpenAI()

In [6]:
# from transformers import BertTokenizer
from transformers import RobertaTokenizer

# load bert tokenizer from huggingface
tokenizer = RobertaTokenizer.from_pretrained(
    'roberta-base'
)

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [7]:
def build_dict(input_batch):
  # store a batch of sparse embeddings
    sparse_emb = []
    # iterate through input batch
    for token_ids in input_batch:
        # convert the input_ids list to a dictionary of key to frequency values
        d = dict(Counter(token_ids))
        tokenids = list(set(token_ids))
        # remove special tokens and append sparse vectors to sparse_emb list
        # sparse_emb.append({key: d[key] for key in d if key not in [101, 102, 103, 0]})
        sparse_emb.append({"indices":tokenids, "values":[float(d[id]) for id in tokenids]})
    # return sparse_emb list
    return sparse_emb

In [7]:
def generate_sparse_vectors(context_batch):
    input_ids = tokenizer(
    context_batch, padding=True, truncation=True,
     max_length=512
)["input_ids"]
    sparse_embeds = build_dict(input_ids)
    return sparse_embeds

In [8]:
pc = Pinecone(api_key= os.getenv("PINECONE_API_KEY_500"))
index_name = "hybrid-rag-roberta"

In [11]:
indices = []
for index in pc.list_indexes():
    indices.append(index["name"])

if index_name in indices :
    print(f"index {index_name} already exists!")
    index = pc.Index(index_name)
else:
    pc.create_index(
  name=index_name,
  dimension=3072,
  metric="dotproduct",
  spec=ServerlessSpec(
    cloud="aws",
    region="us-east-1"
  ),
  deletion_protection="disabled"
)
    index = pc.Index(index_name)
    print(f"index {index_name} created")



index hybrid-rag-roberta already exists!


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=512,
    chunk_overlap=128,
    length_function=len,
    is_separator_regex=False,
)

In [11]:
###PREPARE WEBPAGE DATA 
###Get the full text & Chunks
import json 

with open(r"../../data/url_content_mapping.json", "r",encoding="utf-8") as file:
    data = json.load(file)
texts = []
for content in data:
    texts.extend(text_splitter.create_documents([content["content"]], [{"source":content["url"]}]))


In [12]:
len(texts)

1278

In [54]:
ids = [str(x) for x in range(len(texts))]
meta = [text.metadata.update({"content": text.page_content}) or text.metadata for text in texts]
content = [text.page_content for text in texts]
dense_embeds_request = client.embeddings.create(input=content,
    model="text-embedding-3-large")
dense_embeds = [item.embedding for item in dense_embeds_request.data]
sparse_embeds = generate_sparse_vectors(content)
vectors = []

for _id, sparse, dense, metadata in zip(ids, sparse_embeds, dense_embeds, meta):
        vectors.append({
            'id': _id,
            'sparse_values': sparse,
            'values': dense,
            'metadata': metadata
        })



In [55]:
batch_size = 100

length = len(vectors) // batch_size +1
start=0
end = batch_size+1
for i in range(length):
  print(f"{start} - {end}")
  
  if end != len(vectors):
    index.upsert(vectors=vectors[start:end])
  else:
    index.upsert(vectors=vectors[start:])
  print("upserted _successfully")
  start = end 
  end = min(end +batch_size , len(vectors))


0 - 101
upserted _successfully
101 - 201
upserted _successfully
201 - 301
upserted _successfully
301 - 401
upserted _successfully
401 - 501
upserted _successfully
501 - 601
upserted _successfully
601 - 701
upserted _successfully
701 - 801
upserted _successfully
801 - 901
upserted _successfully
901 - 1001
upserted _successfully
1001 - 1101
upserted _successfully
1101 - 1201
upserted _successfully
1201 - 1278
upserted _successfully


In [9]:
from pinecone_text.hybrid import hybrid_convex_scale
class retriever:
        def __init__(self, embed, index):
             self.embed = embed
             self.index = index
        def get_data(self,query,alpha=1):
            sparse_embedding = generate_sparse_vectors([query])[0]
            dense_embedding=self.embed( model="text-embedding-3-large",input=query).data[0].embedding

            dense_vec, sparse_vec = hybrid_convex_scale(
                 dense_embedding, sparse_embedding, alpha=alpha
            )
            vecs = self.index.query(
            vector=dense_vec,
            sparse_vector = sparse_vec,
            top_k=5,
            includeMetadata=True,
            include_values=True
        )["matches"]
            ids=[] 
            for match in vecs:
                ids.append(match.id)
            data = self.index.fetch(ids)
            docs = []
            for key in data["vectors"]:
                docs.append(data["vectors"][key]["metadata"]["content"])
            return docs


In [12]:
embed = client.embeddings.create
ret= retriever(embed, index)


In [ ]:
# ret.get_data("what is the name of the service for new babies?")

In [13]:
prompt ="Given the provided context, generate a response that is accurate, concise, and strictly aligned with the information retrieved. Ensure the response does not include hallucinations, speculations, or unsupported claims. The response should be neutral, fact-based, and respectful, especially when addressing sensitive or ambiguous topics. If the context provided is insufficient, clearly state that more information is needed. Prioritize safety and relevance, and avoid generating offensive or harmful content. Please the answer should be in paragraph style and sentences. do not introduce lists and bullet points"

class generator:

    def __init__(self, llm):
        self.llm = llm
    
    def generate(self, query, context):
        formatted_context = "\n".join([str(doc) for doc in context])
        response = self.llm.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": prompt},
        {
            "role": "user",
            "content": query+formatted_context
        }
    ]
)
        return response.choices[0].message

In [14]:
gen = generator(client)


In [15]:
class Rag_app:
    def __init__(self,llm,retriever):
        self.retriever = retriever
        self.llm=llm
    @instrument
    def retrieve(self, query: str) -> List[str]:
        """
        Method to handle document retrieval.
        IMPORTANT: The method name 'retrieve' will be used in selectors
        """
        documents = self.retriever.get_data(query)
        return documents
    
    @instrument
    def generate(self, query: str, context: List[str]) -> str:
        """
        Method to handle response generation.
        IMPORTANT: The method name 'generate' will be used in selectors
        """
        formatted_context = "\n".join([str(doc) for doc in context])
        response = self.llm.generate(query ,formatted_context)
        return response
    
    @instrument
    def query(self, question: str) -> Dict:
        """
        Main method that orchestrates the RAG pipeline.
        IMPORTANT: Return keys must match selector paths
        """
        context = self.retrieve(question)
        response = self.generate(question, context)
        
        return response.content

In [16]:
rag_app = Rag_app(gen, ret)
# provider = OpenAI(model_engine="gpt-4o", api_key=os.getenv("OPEN_AI_EVAL_KEY"))

import numpy as np
from trulens.core import Feedback
from trulens.core import Select
from trulens.providers.openai import OpenAI
provider = OpenAI()

# Define a groundedness feedback function
f_groundedness = (
    Feedback(
        provider.groundedness_measure_with_cot_reasons, name="Groundedness"
    )
    .on(Select.RecordCalls.retrieve.rets.collect())
    .on_output()
)
# Question/answer relevance between overall question and answer.
f_answer_relevance = (
    Feedback(provider.relevance_with_cot_reasons, name="Answer Relevance")
    .on_input()
    .on_output()
)

# Context relevance between question and each context chunk.
f_context_relevance = (
    Feedback(
        provider.context_relevance_with_cot_reasons, name="Context Relevance"
    )
    .on_input()
    .on(Select.RecordCalls.retrieve.rets[:])
    .aggregate(np.mean)  # choose a different aggregation method if you wish
)

✅ In Groundedness, input source will be set to __record__.app.retrieve.rets.collect() .
✅ In Groundedness, input statement will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Answer Relevance, input prompt will be set to __record__.main_input or `Select.RecordInput` .
✅ In Answer Relevance, input response will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Context Relevance, input question will be set to __record__.main_input or `Select.RecordInput` .
✅ In Context Relevance, input context will be set to __record__.app.retrieve.rets[:] .


In [61]:
from trulens.apps.custom import TruCustomApp
##NAMING CONVENTION eval-{Retriever}-{generator}-{chunksize}-@{k}

tru_rag = TruCustomApp(
    rag_app,
    app_name="HYBRID RAG",
    app_version="robert-4o-large_3-500-alpha_1.0",
    feedbacks=[f_groundedness, f_answer_relevance, f_context_relevance],
)

In [62]:
from trulens.core.utils.pace import Pace

# Define your desired pacing rate
pace = Pace(marks_per_second=0.5, seconds_per_period=30.0)

In [63]:
with open("../../GroundTruths_Dataset - Sheet1.csv", mode='r', encoding='utf-8') as file:
    csv_reader = csv.DictReader(file)
    # Iterate through rows as dictionaries
    queries = []
    for row in csv_reader:
        queries.append(row["query"]) 
len(queries)

23

In [64]:
with tru_rag as recording:
    for eval in queries:
        print(eval)
        pace.mark()
        rag_app.query(
        eval
    )

How do I register for controlled or semi-controlled drugs custody?
What are the requirements for renewing the registration of a conventional pharmaceutical product?


c:\Users\abdal\Desktop\RAG-main\env\Lib\site-packages\trulens\feedback\llm_provider.py:1521: UserWarning: Failed to process and remove trivial statements. Proceeding with all statements.
  warnings.warn(


How do I appeal a decision made by the Medical Licensing Committee?
What is the process for obtaining a certificate of amendment for registered pharmaceutical products?
How can I get a product classified?
What are the steps to re-license a pharmaceutical facility?
How can I renew my license as a nurse or medical professional?
What's the process for getting a permit to import medical equipment?
How can I renew my health facility license?
What are the steps to register a change in a doctor's professional title?
How do I re-license a health facility after cancellation or suspension?
How do I change the technical director of my private medical facility?
What's the process for re-licensing nurses and medical professionals?
How can I request a list of licensed pharmaceutical facilities in the UAE?
How do I renew my registration certificate to practice nursing or midwifery?
What are the requirement documents for the good standing certificate of medical staff in the sector the is fee-exempt fo

In [23]:
from trulens.dashboard import run_dashboard

run_dashboard(session)

Starting dashboard ...


Accordion(children=(VBox(children=(VBox(children=(Label(value='STDOUT'), Output())), VBox(children=(Label(valu…

Dashboard started at http://192.168.1.12:15908 .


<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>